In [1]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

C:\Users\Vishal\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Vishal\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
ratings = pd.read_csv("../data/processed/ratings_clean.csv")
books = pd.read_csv("../data/processed/books_enhanced.csv")

print(ratings.head())
print(books.head())

   user_id  book_id  rating  rating_normalized
0        1      258       5               1.00
1        2     4081       4               0.75
2        2      260       5               1.00
3        2     9296       5               1.00
4        2     2318       3               0.50
   book_id  goodreads_book_id  best_book_id  work_id  books_count       isbn  \
0        1            2767052       2767052  2792775          272  439023483   
1        2                  3             3  4640799          491  439554934   
2        3              41865         41865  3212258          226  316015849   
3        4               2657          2657  3275794          487   61120081   
4        5               4671          4671   245494         1356  743273567   

         isbn13                      authors  original_publication_year  \
0  9.780000e+12              Suzanne Collins                     2008.0   
1  9.780000e+12  J.K. Rowling, Mary GrandPré                     1997.0   
2  9.780000e

In [3]:
# Keep only high ratings
positive_ratings = ratings[ratings['rating'] >= 4]

# Merge book titles
data = positive_ratings.merge(books[['book_id','title']], on='book_id')

# Group by user → create transactions
transactions = data.groupby('user_id')['title'].apply(list).tolist()

print("Total transactions:", len(transactions))

Total transactions: 12752


In [4]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df = pd.DataFrame(te_array, columns=te.columns_)

In [5]:
item_support = df.mean()

df = df.loc[:, (item_support > 0.02) & (item_support < 0.4)]

print("Items after pruning:", df.shape[1])

Items after pruning: 504


In [6]:
min_support = max(0.03, 100 / len(df))

frequent_itemsets = fpgrowth(
    df,
    min_support=min_support,
    use_colnames=True
)

In [7]:
frequent_itemsets = frequent_itemsets[
    frequent_itemsets['itemsets'].apply(len) <= 3
]

In [8]:
rules = association_rules(
    frequent_itemsets,
    num_itemsets=len(df),   # total transactions
    metric="lift",
    min_threshold=1.2
)

In [9]:
def recommend_books(book_name, rules, top_n=5):
    filtered = rules[
        rules['antecedents'].astype(str).str.contains(book_name)
    ]
    return filtered.sort_values(by='lift', ascending=False).head(top_n)

In [10]:
rules['score'] = (
    0.5 * rules['confidence'] +
    0.5 * rules['lift']
)

rules = rules.sort_values('score', ascending=False)

In [11]:
def recommend_books(book_name, rules, top_n=5):

    # Check if book is in antecedent set
    filtered = rules[
        rules['antecedents'].map(lambda x: book_name in x)
    ]

    if filtered.empty:
        return []

    # Weighted scoring
    filtered['score'] = (
        0.5 * filtered['confidence'] +
        0.5 * filtered['lift']
    )

    filtered = filtered.sort_values('score', ascending=False)

    recommendations = []

    for cons in filtered['consequents']:
        recommendations.extend(list(cons))

    # Remove duplicates
    recommendations = list(dict.fromkeys(recommendations))

    return recommendations[:top_n]

In [12]:
def fallback_popular_books(frequent_itemsets, top_n=5):
    """
    Returns top N most popular books based on support.
    Used for cold start handling.
    """
    popular = frequent_itemsets.sort_values(
        "support", ascending=False
    )
    
    # Extract actual book names from frozenset
    books = []
    for item in popular.head(top_n)['itemsets']:
        books.extend(list(item))
    
    return books[:top_n]


def smart_recommend(book_name, rules, frequent_itemsets, top_n=5):
    """
    Smart recommendation engine:
    - Uses association rules if available
    - Falls back to popular books if no rule found
    """
    
    filtered = rules[
        rules['antecedents'].astype(str).str.contains(book_name)
    ]
    
    if filtered.empty:
        return fallback_popular_books(frequent_itemsets, top_n)
    
    filtered = filtered.sort_values(by='lift', ascending=False)
    
    recommendations = []
    
    for cons in filtered['consequents']:
        recommendations.extend(list(cons))
    
    recommendations = list(dict.fromkeys(recommendations))
    
    return recommendations[:top_n]

In [13]:
def recommend_for_user(user_books, rules, top_n=5):
    
    filtered = rules[
        rules['antecedents'].apply(
            lambda x: any(book in x for book in user_books)
        )
    ]
    
    filtered['score'] = (
        0.4 * filtered['confidence'] +
        0.6 * filtered['lift']
    )
    
    filtered = filtered.sort_values('score', ascending=False)
    
    recommendations = []
    
    for cons in filtered['consequents']:
        recommendations.extend(list(cons))
    
    recommendations = list(set(recommendations) - set(user_books))
    
    return recommendations[:top_n]

In [14]:
import json

def get_recommendation_json(book_name, rules):
    
    recs = recommend_books(book_name, rules)
    
    return json.dumps({
        "input_book": book_name,
        "recommendations": recs
    })

In [15]:
rules['leverage'] = rules['support'] - (
    rules['antecedent support'] * rules['consequent support']
)

rules['final_score'] = (
    0.3 * rules['confidence'] +
    0.4 * rules['lift'] +
    0.3 * rules['leverage']
)

rules = rules.sort_values('final_score', ascending=False)

In [16]:
def trending_books(basket, top_n=5):
    return basket.mean().sort_values(
        ascending=False
    ).head(top_n)

In [17]:
import time
from mlxtend.frequent_patterns import apriori, fpgrowth

start = time.time()
apriori(df, min_support=0.05, use_colnames=True)
ap_time = time.time() - start

start = time.time()
fpgrowth(df, min_support=0.05, use_colnames=True)
fp_time = time.time() - start

print("Apriori Time:", ap_time)
print("FP-Growth Time:", fp_time)

Apriori Time: 1.7609355449676514
FP-Growth Time: 2.467390537261963


In [18]:
import pickle
import os

# Create models folder if not exists
os.makedirs("../models", exist_ok=True)

# Save rules inside models folder
with open("../models/association_rules_model.pkl", "wb") as f:
    pickle.dump(rules, f)

print("Model saved successfully inside models folder.")

Model saved successfully inside models folder.


In [19]:
coverage = len(rules) / len(frequent_itemsets)
print("Rule Coverage:", coverage)

Rule Coverage: 3.9652131308182264


In [20]:
def evaluate_precision(test_users, rules, transactions):
    
    hits = 0
    total = 0
    
    for user in test_users:
        user_books = transactions[user]
        if len(user_books) < 2:
            continue
        
        input_book = user_books[0]
        actual_books = set(user_books[1:])
        
        recs = recommend_books(input_book, rules)
        
        hits += len(set(recs) & actual_books)
        total += len(recs)
    
    return hits / total if total != 0 else 0